In [61]:
import os 
cwd = os.getcwd()
cwd

os.chdir("/Users/jmthomas565/Desktop/Education/Applied Stats MSc/MSc Project/Git Files/msc-project-source-code-files-24-25-jmthomas565/Code/Project Code Pipeline")

new_cwd = os.getcwd() 

new_cwd


'/Users/jmthomas565/Desktop/Education/Applied Stats MSc/MSc Project/Git Files/msc-project-source-code-files-24-25-jmthomas565/Code/Project Code Pipeline'

In [62]:
%run 00_inititial_transformations.ipynb

['2017-07-01', '2017-07-02', '2017-07-03', '2017-07-04', '2017-07-05'] ... ['2022-09-27', '2022-09-28', '2022-09-29', '2022-09-30', '2022-10-01']


In [63]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

In [64]:
X_train.columns

Index(['branded_search_spend', 'nonbranded_search_spend', 'facebook_spend',
       'print_spend', 'ooh_spend', 'tv_spend', 'radio_spend', 'gdp',
       'base_rate', 'UNRATENSA_daily'],
      dtype='object')

In [65]:
def adstock(x, theta):
    return tsa.filters.recursive_filter(x, theta)

#locking in final AdStock scores, on cols that need it
X_train["branded_search_spend_adstock"] = adstock(X_train["branded_search_spend"], [0.9])
X_train["nonbranded_search_spend_adstock"] = adstock(X_train["nonbranded_search_spend"], [0.9])
X_train["facebook_spend_adstock"] = adstock(X_train["facebook_spend"], [0.9])

#Creating the lags for macro variables
X_train['gdp_lag_1'] = X_train['gdp'].shift(-1)
X_train['base_rate_lag_1'] = X_train['base_rate'].shift(-1)
X_train['UNRATENSA_daily_lag_1'] = X_train['UNRATENSA_daily'].shift(-1)
X_train['sales_lag_1'] = y_train['sales'].shift(-1)

#Loading only the columns that are required into the df. 
#Note that the missing offline channels are left out, because the baseline and adstock models found them not required.
X_train_tran = X_train[[
    "branded_search_spend_adstock",
    'nonbranded_search_spend_adstock',
    'facebook_spend_adstock',
    'print_spend',
    'gdp', 
    'base_rate', 
    'UNRATENSA_daily',
    'sales_lag_1',
    'gdp_lag_1', 
    'base_rate_lag_1', 
    'UNRATENSA_daily_lag_1']]

#Setting the dfs from the 2nd row, because the lags have created null rows. 
X_train_tran = X_train_tran.iloc[:-1]

y_train = y_train.iloc[:-1]

In [66]:
def adj_r2(y_true, y_pred, p):
    r2 = r2_score(y_true, y_pred)
    n = len(y_true)
    #Making sure the denominator of adj R^2 > 0
    if n - p - 1 <= 0:
        return np.nan
    calculation = 1 - (1 - r2) * (n - 1) / (n - p - 1)
    return calculation

In [67]:

def optimise_ridge_holdout_scaled(X_train_tran, y_train, alphas):
    
    #Cross-validation hold-out splits
    hold_out_split = -26
    y_train_calib = y_train.iloc[:hold_out_split]
    y_train_hold_out = y_train.iloc[hold_out_split:]

    X_train_calib = X_train_tran.iloc[:hold_out_split]
    X_val_hold_out = X_train_tran.iloc[hold_out_split:]

    scaler = StandardScaler()
    X_train_calib = scaler.fit_transform(X_train_calib)
    X_val_hold_out = scaler.transform(X_val_hold_out)

    n_regressors = X_train_tran.shape[1]

    #Looping through values of alpha (ridge lambda)
    for alpha in alphas:
        ridge = Ridge(alpha=alpha, positive=True)
        ridge.fit(X_train_calib, y_train_calib)
        y_pred = ridge.predict(X_val_hold_out)

        score = adj_r2(y_train_hold_out, y_pred, p=X_train_tran.shape[1])
            
        #Printing the final adj_R^2 values            
        print(f"Alpha:{alpha}: Adjusted R2 = {round(score, 4)}")


#Setting the paramter space
alphas_to_test = [0, 0.01, 0.1, 1, 10, 100, 1000]

ridge_results = optimise_ridge_holdout_scaled(X_train_tran, y_train, alphas_to_test)


Alpha:0: Adjusted R2 = 0.5784
Alpha:0.01: Adjusted R2 = 0.5784
Alpha:0.1: Adjusted R2 = 0.5787
Alpha:1: Adjusted R2 = 0.5809
Alpha:10: Adjusted R2 = 0.597
Alpha:100: Adjusted R2 = 0.4317
Alpha:1000: Adjusted R2 = -0.2797


In [68]:
X_train_tran.columns

Index(['branded_search_spend_adstock', 'nonbranded_search_spend_adstock',
       'facebook_spend_adstock', 'print_spend', 'gdp', 'base_rate',
       'UNRATENSA_daily', 'sales_lag_1', 'gdp_lag_1', 'base_rate_lag_1',
       'UNRATENSA_daily_lag_1'],
      dtype='object')